<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day21_practice1_%EB%B2%A1%ED%84%B0%EA%B2%80%EC%83%89_%EC%9B%90%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# RAG (Retrieval-Augmented Generation, 검색 증강 생성)
# 벡터 검색의 원리
# 문서를 벡터로, 질문도 벡터로
# 코사인 유사도를 가장 가까운 문서 찾기 = 벡터 검색
# 브루트 포스(Brute Force)는 가능한 모든 경우의 수를 하나씩 전부 대입해 보며 정답을 찾는 완전 탐색 알고리즘 : FAISS, Chroma 벡터 DB가 하는 일

원본 문서 ─▶ ① 청킹 ─▶ 문서 조각들 ─▶ ② 임베딩 ─▶ 벡터들 ─▶ ③ 유사도 검색

In [17]:
import torch
import torch.nn.functional as F
import re

In [18]:
# 셀 1. 시청의 도로 규정 : knowledge/ 폴더의 규정 문서들 사용
documents = [
    "심각도 4 이상의 포트홀은 접수 후 24시간 이내에 긴급 보수팀을 배정한다.",
    "심각도 3 이하의 도로 균열은 주간 정기 점검 일정에 포함하여 7일 이내 처리한다.",
    "맨홀 뚜껑 이상 신고는 안전 펜스를 우선 설치하고 상수도사업소에 이관한다.",
    "낙하물 신고는 2차 사고 예방을 위해 즉시 현장 출동을 원칙으로 한다.",
    "처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.",
    "야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다.",
]
print(f"knowledge/: 규정 {len(documents)}개")

knowledge/: 규정 6개


In [19]:
# 셀 2. 문장 → 벡터
# 1) 사전 만들기: 글자 2조각 → 번호, n-gram 임베딩
def make_vocab(texts, n=2):
  grams = set()
  for t in texts:
    t = re.sub(r"\s+", "", t) # \s: 공백 제거
    grams |= {t[i:i+n] for i in range(len(t) - n + 1)} # |=: 합집합, 조각 시작 위치, 모든 2글자 조각을 모은다.
  return {g: i for i, g in enumerate(sorted(grams))}

vocab = make_vocab(documents)
print(vocab)

{'(2': 0, ')접': 1, '06': 2, '22': 3, '24': 4, '2시': 5, '2차': 6, '3이': 7, '4시': 8, '4이': 9, '6시': 10, '7일': 11, '~0': 12, '각도': 13, '간(': 14, '간이': 15, '간정': 16, '검일': 17, '게앱': 18, '고는': 19, '고상': 20, '고예': 21, '고자': 22, '과함': 23, '관한': 24, '균열': 25, '급보': 26, '기점': 27, '긴급': 28, '껑이': 29, '께신': 30, '낙하': 31, '내에': 32, '내처': 33, '는2': 34, '는사': 35, '는안': 36, '는익': 37, '다.': 38, '담당': 39, '당자': 40, '도3': 41, '도4': 42, '도로': 43, '도사': 44, '동을': 45, '된신': 46, '뚜껑': 47, '로균': 48, '로통': 49, '로한': 50, '료된': 51, '를시': 52, '를우': 53, '리를': 54, '리완': 55, '리한': 56, '림으': 57, '맨홀': 58, '물신': 59, '방을': 60, '배정': 61, '보수': 62, '보한': 63, '사고': 64, '사업': 65, '사진': 66, '상수': 67, '상신': 68, '상의': 69, '선설': 70, '설치': 71, '소에': 72, '수도': 73, '수신': 74, '수팀': 75, '수후': 76, '스를': 77, '시)': 78, '시~': 79, '시간': 80, '시작': 81, '시현': 82, '신고': 83, '심각': 84, '안전': 85, '알림': 86, '앱알': 87, '야간': 88, '업소': 89, '에게': 90, '에긴': 91, '에이': 92, '에포': 93, '여7': 94, '열은': 95, '예방': 96, '오전': 97, '완료': 98, '우선': 99, '원칙': 100

In [20]:
# 2) 문장 → 벡터 변환
def embed(text, vocab=vocab, n=2):
  v = torch.zeros(len(vocab))
  t = re.sub(r"\s+", "", text) # 공백이 없는 문장
  for i in range(len(t) - n + 1): # 2글자씩
    g = t[i:i+n]
    if g in vocab:
      v[vocab[g]] += 1 # 단어 등장 횟수 증가
  return v

# 3) 문서 전체를 행렬로
doc_vectors = torch.stack([embed(d) for d in documents])
print(f"문서 벡터: {tuple(doc_vectors.shape)} (문서 6개 × {len(vocab)}차원)")

문서 벡터: (6, 157) (문서 6개 × 157차원)


In [21]:
print(doc_vectors)

tensor([[0., 0., 0., 0., 1., 0., 0., 0., 1., 1., 0., 0., 0., 1., 0., 1., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 0.,
         0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
         0., 0., 0., 1., 1., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
         0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0.,
         0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 1., 0.,
         0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 1., 0., 0., 1., 1.,
         0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.,
         0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
         0., 0., 1., 0., 0., 0., 0., 0., 0., 0.

In [28]:
# 셀 3. 검색 - 질문 벡터와 가장 가까운 문서 (코사인)
def search(query, k=2):
  q = embed(query) # 임베딩 → 벡터화
  sims = F.cosine_similarity(q.unsqueeze(0), doc_vectors) # 코사인 유사도, (157,)→(1,157), 질문 1개 vs 문서 6개를 한번에 유사도 검사
  top = sims.argsort(descending=True)[:k] # 내림차순으로 정렬 상위 k개 인덱스 번호
  return [(documents[i], sims[i].item()) for i in top]

질문들 = [
  "심각도 4짜리 포트홀은 얼마나 빨리 처리되나요?",
  "맨홀 신고하면 어디서 처리해요?",
  "밤에 신고하면 언제 처리 시작돼요?",
]
for q in 질문들:
  print(f"\n질문: {q}")
  for doc, s in search(q):
    print(f"[{s:.3f}] {doc}")

# RAG - Retrieval 검색
# 문서 벡터화(색인) → 질문 벡터화 → 코사인 →상위 k개


질문: 심각도 4짜리 포트홀은 얼마나 빨리 처리되나요?
[0.407] 심각도 4 이상의 포트홀은 접수 후 24시간 이내에 긴급 보수팀을 배정한다.
[0.197] 심각도 3 이하의 도로 균열은 주간 정기 점검 일정에 포함하여 7일 이내 처리한다.

질문: 맨홀 신고하면 어디서 처리해요?
[0.322] 처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.
[0.207] 맨홀 뚜껑 이상 신고는 안전 펜스를 우선 설치하고 상수도사업소에 이관한다.

질문: 밤에 신고하면 언제 처리 시작돼요?
[0.322] 처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.
[0.302] 야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다.


In [ ]:
# 셀 4. FAISS (Facebook AI Similarity Search): 벡터 검색 전용 라이브러리
# 질문 벡터 vs 모든 문서 벡터 계산을 수십억 벡터에서도 하도록 최적화